# E3 — seed stability on CIFAR-100

Reuses the three E1 seed-1 results and runs six new evaluations for seeds 2 and 3 using the supplied seed-2/3 base.pt checkpoints. This notebook does not train backbones. The class-order seed remains 1993.

Before running: attach `e1-seed1-results(3).zip` and the three-seed backbone Dataset as Kaggle Inputs; set their paths below. Enable Internet and a GPU. The notebook reuses E1 seed 1 and the supplied seed-2/3 checkpoints.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys

REPO_URL = 'https://github.com/duongthienphu/fecam-study.git'
REPO_BRANCH = 'feat/e3-seed-stability'
REPO_DIR = Path('/kaggle/working/fecam-study')
if REPO_DIR.exists():
    subprocess.run(['git', 'fetch', 'origin', REPO_BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', '-B', REPO_BRANCH, 'FETCH_HEAD'], cwd=REPO_DIR, check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'datasets', 'PyYAML', 'matplotlib', 'wandb'], check=True)
print(subprocess.run(['git', 'log', '-1', '--oneline'], capture_output=True, text=True, check=True).stdout.strip())

In [ ]:
# Attach e1-seed1-results(3).zip as a Kaggle Dataset, or upload its extracted e1-seed1 folder.
import json, torch, zipfile
E1_ROOT = Path('/kaggle/input/e1-seed1-results/e1-seed1')
if not (E1_ROOT / 'ncm/summary.json').is_file():
    zip_candidates = [p for p in Path('/kaggle/input').rglob('*.zip') if p.name.startswith('e1-seed1-results')]
    if len(zip_candidates) != 1:
        raise FileNotFoundError('Attach the E1 results ZIP once, or set E1_ROOT to its extracted e1-seed1 directory.')
    extracted = Path('/kaggle/working/e1-seed1-input')
    with zipfile.ZipFile(zip_candidates[0]) as archive:
        archive.extractall(extracted)
    E1_ROOT = extracted / 'e1-seed1'
SEED1_SUMMARIES = [
    E1_ROOT / 'ncm/summary.json',
    E1_ROOT / 'fecam_shared/summary.json',
    E1_ROOT / 'fecam_classwise/summary.json',
]
for path in SEED1_SUMMARIES:
    if not path.is_file():
        raise FileNotFoundError(f'Missing completed E1 seed-1 summary: {path}')
if not __import__('torch').cuda.is_available():
    raise RuntimeError('Enable a GPU accelerator in Kaggle Notebook Settings')

OUTPUT_ROOT = Path('/kaggle/working/e3')
# Dataset containing backbone-seed1/base.pt, backbone-seed2/base.pt, and backbone-seed3/base.pt.
BACKBONE_DATASET_ROOT = Path('/kaggle/input/datasets/phuonganh31/resnet18-backbone-with-3-seeds')
CHECKPOINT_TEMPLATE = str(OUTPUT_ROOT / 'backbones/seed{seed}/base.pt')
for seed in (2, 3):
    source = BACKBONE_DATASET_ROOT / f'backbone-seed{seed}' / 'base.pt'
    destination = OUTPUT_ROOT / 'backbones' / f'seed{seed}'
    destination = destination / 'base.pt'
    if not source.is_file():
        raise FileNotFoundError(f'Attach the three-seed backbone Dataset and check this path: {source}')
    destination.parent.mkdir(parents=True, exist_ok=True)
    if not destination.is_file():
        shutil.copy2(source, destination)
    checkpoint = torch.load(destination, map_location='cpu', weights_only=True)
    metadata = checkpoint['metadata']
    checkpoint_config = metadata['config']
    reference_config = json.loads(SEED1_SUMMARIES[0].read_text(encoding='utf-8'))['config']
    for key in ('protocol', 'dataset', 'split', 'backbone', 'training'):
        if checkpoint_config.get(key) != reference_config.get(key):
            raise ValueError(f'Seed-{seed} checkpoint/config mismatch: {key}')
    if checkpoint_config.get('seed') != seed:
        raise ValueError(f'Expected training seed {seed}, got {checkpoint_config.get("seed")}')
    if metadata.get('class_order') != json.loads(SEED1_SUMMARIES[0].read_text(encoding='utf-8'))['class_order']:
        raise ValueError(f'Seed-{seed} checkpoint has a different class order')
    print(f'Validated supplied seed-{seed} checkpoint: {destination}')

In [ ]:
command = [
    sys.executable, 'scripts/run_e3.py',
    '--seed1-summaries', *[str(path) for path in SEED1_SUMMARIES],
    '--checkpoint-template', CHECKPOINT_TEMPLATE,
    '--output-dir', str(OUTPUT_ROOT),
    '--device', 'cuda:0',
]
subprocess.run(command, check=True)

all_summaries = [str(path) for path in SEED1_SUMMARIES]
for seed in (2, 3):
    all_summaries.extend(str(OUTPUT_ROOT / 'evaluations' / f'seed{seed}' / method / 'summary.json')
                        for method in ('ncm', 'fecam_shared', 'fecam_classwise'))

# Optional W&B upload using a Kaggle Secret named WANDB_API_KEY.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('WANDB_API_KEY')
except Exception:
    print('WANDB_API_KEY secret unavailable; skipping W&B upload.')
else:
    subprocess.run([sys.executable, 'scripts/upload_e3_to_wandb.py', *all_summaries, '--report', str(OUTPUT_ROOT / 'report/e3_seed_stability.csv'), '--plot', str(OUTPUT_ROOT / 'report/e3_seed_stability.png')], check=True)

shutil.make_archive('/kaggle/working/e3-results-and-checkpoints', 'zip', root_dir='/kaggle/working', base_dir='e3')
print('Download e3-results-and-checkpoints.zip from Kaggle Output. It contains evaluation results and resumable backbone checkpoints.')